In [ ]:

import pandas as pd
import numpy as np  # BUG FIX: convention alias is 'np', not 'nm'. np.nan is the standard NaN constant.

# NOTE on NaN vs NULL:
# Pandas uses NaN (float) internally for missing values — there is no "real NULL" in pandas.
# np.nan IS NaN. pd.isna() / pd.isnull() catch both NaN and None. You don't need numpy just for that.

test_data = pd.read_csv("ArithmeticError.csv")


# ── CLEANING THE HEADER ──────────────────────────────────────────────────────

# QUESTION: "why do you need .str after like every word lol?"
#
# ANSWER: test_data.columns is a pandas Index object, NOT a plain Python string.
# To use string methods (strip, replace, lower, etc.) on an Index or Series,
# pandas requires you to go through the .str "accessor" first.
# Think of .str as a gateway that says "apply this string method to every element".
#
# Each method call returns a NEW Index/Series, so you chain another .str to keep going:
#   test_data.columns             → Index object
#   test_data.columns.str.strip() → new Index (whitespace removed from each name)
#   ...str.replace(')', '')       → new Index (')' removed)
#   ...str.lower()                → new Index (all lowercase)
#
# BUG FIX: .lower() is a string method, so it also needs .str before it:
test_data.columns = test_data.columns.str.strip().str.replace(')', '').str.lower()

print(test_data.head())


# ── SELECTING COLUMNS ────────────────────────────────────────────────────────

# BUG FIX: 'test_date' → 'test_data'  (typo — Python is case/spelling sensitive)
#
# LESSON on [] vs ():
#   df['col']          → single brackets select ONE column → returns a Series (1D)
#   df[['col1','col2']]→ double brackets select MULTIPLE columns → returns a DataFrame (2D)
#
# The outer [] is the standard Python "subscript" (index) operator on the DataFrame.
# The inner [] is just a Python list literal. So df[['a','b']] == df[ ['a','b'] ]
# You're passing a LIST into the subscript operator, not using double-brackets as special syntax.
test_data = test_data[['id', 'text', 'label', 'date', 'indicator']]


# ── MISSING VALUES ────────────────────────────────────────────────────────────

# .isna() returns a DataFrame of True/False for every cell.
# .sum() then counts the Trues (since True == 1) per column.
print(test_data.isna().sum())

# dropna(subset=['label']) only drops rows where 'label' is NaN — leaves other NaNs alone.
# We reassign back to test_data because dropna() returns a NEW DataFrame, it does NOT edit in place.
# (Most pandas methods are non-destructive by design.)
test_data = test_data.dropna(subset=['label'])


# ── str.contains — checking for special characters ───────────────────────────

# BUG FIX: 'r'\$' or r'\n'' does NOT work the way you think.
# Python evaluates   r'\$' or r'\n'   as plain Python "or" between two strings.
# In Python, a non-empty string is truthy, so   'anything' or 'something'   always returns
# the first string — equivalent to just r'\$'. The second pattern is silently ignored.
#
# To match EITHER pattern in regex, use the pipe | inside ONE regex string:
test_data_new = test_data['text'].astype(str).str.contains(r'\$|\n')
# r'\$|\n' means: literal dollar-sign  OR  newline character

print(test_data_new)
# This prints a Series of True/False — one value per row.


# ── HOW DOES str.replace WITH REGEX REALLY WORK? ─────────────────────────────

# ANSWER: Yes, it uses Python's re (regex) module under the hood when regex=True.
# r'[^d.-]' means "match any character that is NOT d, period, or dash".
#
# BUG FIX: You almost certainly meant r'[^\d.-]' (note the backslash before d).
# \d = any digit (0-9).  Without the backslash, d just means the literal letter "d".
# So the corrected pattern: "keep only digits, dots, and dashes — remove everything else"
test_data['text'] = test_data['text'].astype(str).str.replace(r'[^\d.-]', '', regex=True)


# ── RECASTING A COLUMN TO NUMERIC ─────────────────────────────────────────────

# BUG FIX: pd.to_numeric(test_data['test']) → wrong column name 'test', should be 'text'
# errors='coerce' turns anything that can't convert into NaN instead of crashing.
test_data['text'] = pd.to_numeric(test_data['text'], errors='coerce')


# ── DROP DUPLICATES ───────────────────────────────────────────────────────────

# BUG FIX: drop_duplicates() returns a NEW DataFrame — it does NOT edit in place.
# Without reassigning, the line below does nothing (result is thrown away).
test_data = test_data.drop_duplicates(subset=["name"])


# ── REPLACE VALUES WITH A MAPPING DICT ───────────────────────────────────────

print(test_data['label'].unique())

desired = {
    "samplEing": "sample",
    "hEllo":     "hello"
}

# QUESTION: "what about values not in the dict?"
# ANSWER: .replace() leaves unmatched values UNCHANGED. ✓ Your assumption was correct.
# .map() is different — it turns unmatched values into NaN. Avoid .map() for this use case.
test_data['label'] = test_data['label'].replace(desired)


# ── MAP BOOLEANS ──────────────────────────────────────────────────────────────

print(test_data['indicator'].unique())

bool_map = {
    "yes":  True,
    "Ye":   True,
    "Naaa": False
}

# BUG FIX: 'false' → False  (Python booleans are capitalized: True / False)
# .map(bool_map) applies the dict — values NOT in the dict become NaN.
# .fillna(False) replaces those NaNs with False before casting to bool.
test_data['indicator'] = test_data['indicator'].map(bool_map).fillna(False).astype(bool)


# ── DATA TYPES IN PANDAS ──────────────────────────────────────────────────────

# QUESTION: "what are the different data types in pandas?"
# Common dtypes you'll see from .dtype or .dtypes:
#   int64      → integer numbers
#   float64    → decimal numbers (also used for columns with NaN, since NaN is a float)
#   object     → strings (or mixed types) — the catch-all
#   bool       → True/False
#   datetime64 → dates and datetimes (after pd.to_datetime)
#   category   → a fixed set of repeated string values (like an enum — saves memory)

# BUG FIX: 'df' is undefined — variable is called 'test_data'
print(test_data["date"].dtype)


# ── CONVERTING TO DATETIME ────────────────────────────────────────────────────

# QUESTION: "whats this converting to?"
# ANSWER: pd.to_datetime() converts a string/object column into dtype datetime64[ns].
# Once it's datetime64, you can extract parts like .dt.year, .dt.month, .dt.strftime().
#
# QUESTION: "how to get date_part(col, 'yyyy-mm')?"
# ANSWER: use .dt.strftime('%Y-%m') to produce a string like "2024-03",
#         or .dt.to_period('M') to get a Period object for month-level grouping.
#
# BUG FIX 1: 'pd.tp_datetime' → 'pd.to_datetime'  (typo)
# BUG FIX 2: 'df' → 'test_data'
# BUG FIX 3: 'start_Date' → 'date'  (that's the column we have)
test_data['date'] = pd.to_datetime(test_data['date'], errors='coerce')

# To get 'yyyy-mm' as a string column:
test_data['year_month'] = test_data['date'].dt.strftime('%Y-%m')


# ── BOOLEAN INDEXING WITH .loc ────────────────────────────────────────────────

# BUG FIX: 'test_date' → 'test_data'
conditional_data = test_data["label"].isin(['real', 'fake'])

# QUESTION: "is this the correct syntax? do i need to create an index first?"
# QUESTION: "how does this really work? compare the true/false at row level with standard index?"
#
# ANSWER: You are exactly right in your intuition.
# .isin(['real','fake']) returns a boolean Series — one True/False per row:
#   row 0 → True  (label is 'real')
#   row 1 → False (label is 'spam')
#   row 2 → True  (label is 'fake')   ... etc.
#
# .loc[boolean_series] then uses that True/False mask to filter rows.
# "Keep the row if the mask value at that row's index position is True."
# You do NOT need to create a separate index — loc aligns on the existing DataFrame index automatically.
#
# LESSON on [] vs () vs .loc[] — the big picture:
#
#   df['col']          → [] is the subscript operator. Selects a column by name. Returns Series.
#   df[['a','b']]      → passing a list inside subscript. Selects multiple columns. Returns DataFrame.
#   df[mask]           → passing a boolean Series/array inside subscript. Filters ROWS.
#   df.loc[mask]       → same row filtering, but loc is more explicit and also lets you
#                        select columns: df.loc[mask, 'col'] or df.loc[mask, ['a','b']]
#   df.loc[row, col]   → label-based: row label, column label
#   df.iloc[row, col]  → position-based: row number, column number
#
#   ()  are only used when CALLING a function/method: .dropna(), .head(), .sum(), etc.
#       They are NOT used for selecting data.
#
# Rule of thumb:
#   - Use []  to index / select data (columns, rows by mask)
#   - Use ()  to call methods and functions
#   - Use .str before any string method on a Series or Index
#   - Use .dt before any date method on a datetime Series

print(test_data.loc[conditional_data])
